# Build Android APK with Buildozer on Google Colab

This notebook compiles your Kivy app into an Android APK.

**Instructions:**
1. Runtime → Restart runtime... (if re-running)
2. Runtime → Run all
3. Wait 30-60 minutes
4. Download the APK from the final step


In [ ]:
# Step 1: Install system dependencies
import subprocess, sys, os

print("Installing system dependencies...")
subprocess.run(["apt", "update"], check=True)
subprocess.run([
    "apt", "install", "-y",
    "python3-pip", "python3-dev", "python3-venv", "git", "zip", "unzip",
    "openjdk-17-jdk", "libbz2-dev", "libncurses5-dev", "libffi-dev",
    "libreadline-dev", "libsqlite3-dev", "zlib1g-dev", "liblzma-dev",
    "autoconf", "libtool", "pkg-config", "python3-setuptools",
    "wget", "curl", "build-essential"
], check=True)

local_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = local_bin + ":" + os.environ["PATH"]

print("\nInstalling Cython and Buildozer...")
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "cython==0.29.19"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "buildozer==1.5.0"], check=True)

print("\n✅ Step 1 complete!")

In [ ]:
# Step 2: Patch buildozer source to skip root check (write to disk)
import os, re, buildozer

pkg_dir = os.path.dirname(buildozer.__file__)
init_path = os.path.join(pkg_dir, "__init__.py")
print(f"Patching: {init_path}")

with open(init_path, 'r') as f:
    lines = f.readlines()

new_lines = []
skip = False
for i, line in enumerate(lines):
    if 'def check_root(self):' in line:
        skip = True
        new_lines.append(line)
        new_lines.append('        """Patched: skip root check"""\n')
        new_lines.append('        return\n')
        continue
    if skip:
        if line.startswith('    def ') or line.startswith('class '):
            skip = False
        else:
            continue
    new_lines.append(line)

with open(init_path, 'w') as f:
    f.writelines(new_lines)

print("✅ Patched buildozer source: check_root() disabled")

In [ ]:
# Step 3: Pre-install Android SDK (so buildozer doesn't fail at it)
import os, subprocess

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
os.makedirs(sdk_root, exist_ok=True)
os.environ["ANDROID_SDK_ROOT"] = sdk_root
os.environ["ANDROID_HOME"] = sdk_root

cmdline_dir = os.path.join(sdk_root, "cmdline-tools")
os.makedirs(cmdline_dir, exist_ok=True)

if not os.path.exists(os.path.join(cmdline_dir, "latest")):
    print("Downloading Android command-line tools...")
    subprocess.run([
        "wget", "-q",
        "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip",
        "-O", "/tmp/cmdline-tools.zip"
    ], check=True)
    subprocess.run(["unzip", "-q", "-o", "/tmp/cmdline-tools.zip", "-d", cmdline_dir], check=True)
    os.rename(os.path.join(cmdline_dir, "cmdline-tools"), os.path.join(cmdline_dir, "latest"))
    print("✅ Command-line tools installed.")
else:
    print("✅ Command-line tools already present.")

sdkmanager = os.path.join(cmdline_dir, "latest", "bin", "sdkmanager")
print(f"sdkmanager exists: {os.path.exists(sdkmanager)}")

In [ ]:
# Step 4: Accept SDK licenses and install build-tools 30.0.3
import os, subprocess

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
sdkmanager = f"{sdk_root}/cmdline-tools/latest/bin/sdkmanager"

os.environ["ANDROID_SDK_ROOT"] = sdk_root
os.environ["ANDROID_HOME"] = sdk_root
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Accepting SDK licenses...")
subprocess.run(
    f"echo 'y' | {sdkmanager} --licenses",
    shell=True, capture_output=False
)
print("✅ Licenses accepted.\n")

print("Installing build-tools;30.0.3 and platform;android-30...")
subprocess.run([sdkmanager, "build-tools;30.0.3", "platforms;android-30"], check=True)
print("✅ SDK components installed.")

In [ ]:
# Step 5: Clone repo and fix buildozer.spec
import os, subprocess, shutil

if os.path.exists("network-scanner"):
    shutil.rmtree("network-scanner")

print("Cloning repository...")
subprocess.run(["git", "clone", "https://github.com/Loongood666/network-scanner.git"], check=True)
os.chdir("network-scanner/android")
print(f"Working dir: {os.getcwd()}")

# Fix buildozer.spec: set build_tools version
with open("buildozer.spec", 'r') as f:
    spec = f.read()

if 'android.build_tools' not in spec:
    spec += '\nandroid.build_tools = 30.0.3\n'
else:
    spec = re.sub(r'android\.build_tools\s*=\s*\S+', 'android.build_tools = 30.0.3', spec)

with open("buildozer.spec", 'w') as f:
    f.write(spec)

print("\n✅ buildozer.spec updated.")
print("\nFiles in android/:")
subprocess.run(["ls", "-la"])

In [ ]:
# Step 6: Build APK (30-60 minutes)
import os, subprocess, sys

os.chdir("/content/network-scanner/android")

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
os.environ["ANDROID_SDK_ROOT"] = sdk_root
os.environ["ANDROID_HOME"] = sdk_root

print("Starting APK build...")
print("This will take 30-60 minutes. Please be patient.\n")

result = subprocess.run(
    [sys.executable, "-m", "buildozer", "-v", "android", "debug"],
    capture_output=False,
    timeout=None
)

if result.returncode == 0:
    print("\n✅ Build completed successfully!")
else:
    print(f"\n❌ Build failed with return code {result.returncode}")
    sys.exit(1)

In [ ]:
# Step 7: Download the APK
import os
from google.colab import files

apk_dir = "/content/network-scanner/android/bin"
if os.path.exists(apk_dir):
    apks = [f for f in os.listdir(apk_dir) if f.endswith(".apk")]
    if apks:
        print(f"✅ Found APK: {apks[0]}")
        files.download(os.path.join(apk_dir, apks[0]))
    else:
        print("❌ No APK found in bin/.")
        print("Files in bin/:", os.listdir(apk_dir))
else:
    print("❌ bin/ directory not found.")